### NanoGPT Implementation
- Implementation of Karpathy's lecture on building GPT from scratch: https://www.youtube.com/watch?v=kCc8FmEb1nY&list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ&index=10

- Google collab from lecture: https://colab.research.google.com/drive/1JMLa53HDuA-i7ZBmqV7ZnA3c_fvtXnx-?usp=sharing#scrollTo=O6medjfRsLD9


### Get input data

In [1]:
# import urllib.request

# url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
# urllib.request.urlretrieve(url, "input.txt")

### Build data processing and model in pieces

In [2]:
# read it in to inspect it
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [3]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [4]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [5]:
# unique characters found in text
chars = sorted(set(text))
vocab_size = len(chars)

print("".join(chars))


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [6]:
# map chars to integers and vice versa
stoi = {st: i for i, st in enumerate(chars)}
itos = {i: st for i, st in enumerate(chars)}

# functions to encode and decode a given sequence using stoi and itos mapping
encode = lambda x: [stoi[c] for c in x]
decode = lambda x: "".join([itos[i] for i in x])

print(encode("my name is zeal"))
print(decode(encode("my name is zeal")))

[51, 63, 1, 52, 39, 51, 43, 1, 47, 57, 1, 64, 43, 39, 50]
my name is zeal


In [7]:
# train and test splits
import torch

data = torch.tensor(encode(text), dtype=torch.long)

# 90% training and 10% validation
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Data size: Train = {len(train_data)} | Val = {len(val_data)}")

Data size: Train = 1003854 | Val = 111540


In [8]:
# extract one block of training examples
# a single block contains block_size number of examples

block_size = 8  # same as context_length

x = train_data[:block_size]  # single block
y = train_data[1 : block_size + 1]


print("Single block of example packs block_size number of examples")
for i in range(block_size):
    context = x[: i + 1]
    target = y[i]
    print(f"Example {i} --> context = {context} and target = {target} ")


Single block of example packs block_size number of examples
Example 0 --> context = tensor([18]) and target = 47 
Example 1 --> context = tensor([18, 47]) and target = 56 
Example 2 --> context = tensor([18, 47, 56]) and target = 57 
Example 3 --> context = tensor([18, 47, 56, 57]) and target = 58 
Example 4 --> context = tensor([18, 47, 56, 57, 58]) and target = 1 
Example 5 --> context = tensor([18, 47, 56, 57, 58,  1]) and target = 15 
Example 6 --> context = tensor([18, 47, 56, 57, 58,  1, 15]) and target = 47 
Example 7 --> context = tensor([18, 47, 56, 57, 58,  1, 15, 47]) and target = 58 


In [9]:
# data loader
block_size = 8
batch_size = 4


# output dimension: (B,T) where B (batch dim) is batch_size, T (time dim) is block_size
def get_batch(split):
    data = train_data if split == "train" else val_data
    # randomly pick one index per batch.
    batch_idx = torch.randint(low=0, high=len(data) - block_size, size=(batch_size,))
    # extract each batch of examples that start at their respective idx and end at idx+block_size
    x = torch.stack([data[ix : ix + block_size] for ix in batch_idx])
    y = torch.stack([data[ix + 1 : ix + block_size + 1] for ix in batch_idx])
    return x, y


xb, yb = get_batch(split="train")

print(f"Inputs: {x.shape} \n {x}")
print(f"Targets: {y.shape} \n {y}")

# -------------------------------------------------------------------------------------------------
# visualize every example packed in these four batches for our DECODER transformer block
eg = 0
for batch in range(batch_size):
    for time in range(block_size):
        context = xb[batch, 0 : time + 1]
        target = yb[batch, time]
        print(f"Example {eg} --> context = {context} and target = {target}")
        eg += 1


Inputs: torch.Size([8]) 
 tensor([18, 47, 56, 57, 58,  1, 15, 47])
Targets: torch.Size([8]) 
 tensor([47, 56, 57, 58,  1, 15, 47, 58])
Example 0 --> context = tensor([51]) and target = 53
Example 1 --> context = tensor([51, 53]) and target = 57
Example 2 --> context = tensor([51, 53, 57]) and target = 58
Example 3 --> context = tensor([51, 53, 57, 58]) and target = 1
Example 4 --> context = tensor([51, 53, 57, 58,  1]) and target = 46
Example 5 --> context = tensor([51, 53, 57, 58,  1, 46]) and target = 59
Example 6 --> context = tensor([51, 53, 57, 58,  1, 46, 59]) and target = 51
Example 7 --> context = tensor([51, 53, 57, 58,  1, 46, 59, 51]) and target = 40
Example 8 --> context = tensor([53]) and target = 53
Example 9 --> context = tensor([53, 53]) and target = 56
Example 10 --> context = tensor([53, 53, 56]) and target = 1
Example 11 --> context = tensor([53, 53, 56,  1]) and target = 54
Example 12 --> context = tensor([53, 53, 56,  1, 54]) and target = 56
Example 13 --> context 

In [10]:
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(1266)

In [11]:
# start with bigram model. Bigram model uses a (vocab_size, vocab_size) embedding matrix
# passing "x" of size (3,4) to this matrix --> bigram_embedding_matrix(x) --> it will return a (3,4,vocab_size) output
# where, each element of x is now embedded into vocab_size dimensions


class BigramModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, x, targets=None):
        # x is of size (B,T), where each element is a character index from the vocabulary
        # target is of size (B,T)

        logits = self.token_embedding_table(x)  # (B,T,vocab_size) after passing through the embedding table

        # loss
        if targets == None:
            loss = None
        else:
            # pytorch requires shape (B,C,T) instead of (B,T,C), which is confusing, so we combine the first two dims.
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, x, max_new_tokens):
        # generate one token at a time
        for i in range(max_new_tokens):
            # forward pass through the model
            logits, _ = self.forward(x)  # logits is (B,T,vocab_size)
            # focus only on the last timestep; no dependence on past here
            logits = logits[:, -1, :]  # (B,vocab_size)
            # convert logits to probabilities
            probs = F.softmax(logits, dim=-1)
            # sample next token from probability distribution
            next_token = torch.multinomial(probs, num_samples=1)  # (B,1)
            # append next_token to the original context
            x = torch.cat((x, next_token), dim=1)  # (B,T+1)
        return x


# call the model
model = BigramModel()
logits, loss = model(x=xb, targets=yb)
print(f"Loss = {loss}")

model_output = model.generate(x=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=1000)
print(decode(model_output[0].tolist()))

Loss = 4.478804111480713

X,!okYpX$,HhiGLl,Umvr!$:kOJWsFvoEMxGfn'?cenUkb-diHnC
yJWldGLa ga-SC:
XYgDJ-aEiH-HnRltsz,JCkWt''KboURmS?TLlz.hRu3zv!Cuytc;.Zz,ya?T.$!G.Fvv-3l3G&ZiHolZnursfRdmFu&
XMp;$,IFvkJorY!OWl'Bn,HkzexTVCmFZdsaEMCmltzrybj
ITnCPkZzR
ac G:X;t
d3lnaJJv
WM::pALf-!$Rud?crnBUogeWlm!zReU;CP!HpXphfRpUExCMBLPTBcRd?TUNCkWidPw:$?&beit.SaupvjKbfM:3lt
VhOrJrciSE.zWN:snC$sB,UN,E:fB.SblzRovry-'FQYITq,.s!UPkdmyTse'!OCMw3ltNc b3lcTXzo-vyMB$hVdvdHrpe3lX?Ts:3CobQNcRB yNcAQ,.QW?B::cwgFkWnRuVoclz ,fR$:?xmJy
XQPX:PB3k;CYwHlEJ-3lgZsWiGrlwQHjXF$exBvxtTnC-qr&.'trTfoEiJ3LAB!k 'hPf,n sha,diHZ!dwruUo aHrG 
,hHhRESKN$scXojZByJ-SFhY BfzKY;w:-e UgZjTBsDJ-&QPqiHo!OCT!PW?ZhgBsF,yN bQKAc'?cUemb$saokWr ym diDnRj::cGsg,JWts$Zpyo&CWRbfzDbQ.Gp;sDAnhTo ,FOhoXABO'VAzM:asas$ZNKjwX-fn:?xTsrrZ&!nB,DdmUrxfsgS3!xbHEN& kJUhAGYw3f3rw:M3wJ
zStkc$XTsQH-K;wLkF.TsfhKnpAKTTl;fBPk3CPm!$Ds
.VAFZDqMCDlv-jhTsPyc-epLlhhw3n&Sq;Bkj'zVAN-kAGokglAhPpedmb,OxmyPfwD$hhGLDP.OvdpvmVS:.FvifE3LlEo.SdHnoQmb;pq''B3HG hgDxmCbqj B;eNqxvUedmBS:

In [12]:
# train the bigram model

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for steps in range(1000):
    xb, yb = get_batch("train")

    _, loss = model(x=xb, targets=yb)
    # set all gradients None
    optimizer.zero_grad(set_to_none=True)
    # run backward pass
    loss.backward()
    # make gradient update for each parameter
    optimizer.step()

print(loss.item())


3.7831854820251465


In [13]:
model_output = model.generate(x=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=1000)
print(decode(model_output[0].tolist()))


CDif?y,JQWhHJyf!Yh.SyiSEMsL:foT3lonRZQug:VjhRrTEF&TEDpXorHZ!l,lzoSAz&qxuSxtxthgri?b$Lld;s:iU
DD:rceNk
WjhHHa loCMjhP.GqWv'M.
Xaq:qdscTExDnPJTmFltenODWlQ?xYi:q,aloui:hVCbqOJ!XkkWkITUorf cKAhiovr
p;AAB,VmHIueITt:::rdu.PjK!v yJTgpjjhv'hdmmbbExYLcRDUYekB,JCpe$-uB,titimX$dVNcANzbqOnRuw$W?A-z$P;!$J:fpJAMOhPkWxuSOoifasYUPs
-Mq'VltiVFQPDTIT'xb$xjIUG.n
d Ad'MjQSs$xbF&nNXzMB-SLvUoLYjdceoX.CQB3laa
ZH-mHFOClsXkt'BYhBTIANGQXAQxG:b$MBEUR!$gBype
TnorZLL
By,Vd m.ivbO.PROHz!$.hgr!nCADiMscPOliHay:nk;sMkWvhVJrC Pj'Bkdw-B&Ex,fuJR
W$haVmB,ltY;wIKTnqmgVoQRNjd PNefBfQQn;$ZliZCBQE3GDCZ;SstiH AejkWisOysFQAIs,SbuwOFI'
A,Nnnxg.CNJC-eyslBeflzoQ ROayuyn;H y.H'B.VoP-
3yOMkx
T;X

TG ?ee;OjkAROxJQJ3CMIqaeOe
f osiHaEFQPRZPfo XnRsqY:cnf m;tlgSzz.CivrkWQ3?US$DRrifYck kGRuvlHvof
AQBtaF'uJTBok.as'e$MBJNjITq,Uf:ERd$Z!OQeroQN IT$hrtc'V!xKXa;ookFBUkdPompXz-C?TTgB;YhP3
CcLOJWPMB3llzomFQ&.NJRWdZ g WvwrLL&F;YcXLlllFZf;B$hyTQ$;wXw
WpUIkcTpXkMa
dITtqqi.ely!PX.ZXUJGGpdl!
wqOepbqOutLru&gpvTe,Uo
bTDY!$eOxwQfEJcTwU&
a-UDEhq
deOxvk-e

### Adding positional encoding and single self-attention head with ffn

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(1266)

# ---------------------------Get data ready---------------------------
# read input data
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

# unique characters found in text
chars = sorted(set(text))
vocab_size = len(chars)

# map chars to integers and vice versa
stoi = {st: i for i, st in enumerate(chars)}
itos = {i: st for i, st in enumerate(chars)}
# functions to encode and decode a given sequence using stoi and itos mapping
encode = lambda x: [stoi[c] for c in x]
decode = lambda x: "".join([itos[i] for i in x])

# train-val split: 90% training and 10% validation
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# ---------------------------hyperparameters---------------------------
# hyperparameters
block_size = 8
batch_size = 4
lr = 1e-3
n_embd = 64
head_size = 8
max_iters = 10000
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
# ---------------------------------------------------------------------


# data loader
# output dimension: (B,T) where B (batch dim) is batch_size, T (time dim) is block_size
def get_batch(split):
    data = train_data if split == "train" else val_data
    # randomly pick one index per batch.
    batch_idx = torch.randint(low=0, high=len(data) - block_size, size=(batch_size,))
    # extract each batch of examples that start at their respective idx and end at idx+block_size
    x = torch.stack([data[ix : ix + block_size] for ix in batch_idx])
    y = torch.stack([data[ix + 1 : ix + block_size + 1] for ix in batch_idx])
    x, y = x.to(device), y.to(device)
    return x, y


# add single self attention head to bigram model
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # tril is pre-computed for a head as a fixed-size lower triangular matrix of size (block_size, block_size)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape  # here, C is n_embd --> (B,T,n_embd)
        q = self.query(x)  # (B,T,head_size)
        k = self.key(x)  # (B,T,head_size)
        v = self.value(x)  # (B,T,head_size)

        # calculate scaled weight matrix by wei by head_size
        wei = q @ k.transpose(-2, -1) * (k.size(-1) ** -0.5)  # (B,T,head_size) @ (B,head_size,T) --> (B,T,T)

        # masking the weight matrix
        # during training or inference, your input batch might have a sequence length T
        # that is shorter than block_size (for example, T=16 while block_size = 1024)
        # hence we use self.tril[:T,:T]==0 and not self.tril==0
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))  # (B,T,T)
        # softmax
        wei = F.softmax(wei, dim=-1)  # (B,T,T)

        # compute output
        out = wei @ v  # (B,T,head_size)
        return out


# main model class
class BigramModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # self attention head
        self.sa_head = Head(head_size=head_size)
        self.lm_head = nn.Linear(head_size, vocab_size)  # projection layer from head_size -> vocab_size

    def forward(self, x, targets=None):
        B, T = x.shape
        # information and positional encoding of input are computed and added
        tok_emb = self.token_embedding_table(x)  # (B,T,n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=x.device))  # (T,n_embd)
        x = tok_emb + pos_emb  # (B,T,n_embd)

        # single self-attention head
        x = self.sa_head(x)  # (B,T,head_size)
        # FFN
        # why FFN here: if we pass head's output directly to cross-entropy it will error out because
        # head's output is in range 0 to head_size, where as target is in range 0 to vocab_size.
        # what does it mean: (B,T,vocab_size) --> stores logit values for all vocabulary tokens at each position.
        logits = self.lm_head(x)  # (B,T,vocab_size)

        # loss
        if targets == None:
            loss = None
        else:
            # pytorch requires shape (B,C,T) instead of (B,T,C), which is confusing, so we combine the first two dims.
            # Remember: logits (B,T,vocab_size) matrix stores logit values for all vocabulary tokens at each position.
            # Cross-entropy converts logits to probabilities for each vocabulary token at every position.
            # It then selects the probability of the target token to compute the loss.
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, x, max_new_tokens):
        """
        Generates new tokens using past context.

        This function predicts next tokens one by one up to max_new_tokens.
        It crops the input context to the maximum block size before each step.

        Parameters:
            x (torch.Tensor): Tensor of shape (B, T) with current token indices.
            max_new_tokens (int): The number of new tokens to generate.

        Returns:
            torch.Tensor: Tensor of shape (B, T + max_new_tokens) with full sequence.
        """
        for _ in range(max_new_tokens):
            # limit context to latest block_size tokens
            x_cond = x[:, -block_size:]
            # forward pass this temporary slice through the model
            logits, _ = self.forward(x_cond)  # logits is (B,T,vocab_size)
            # focus only on the last timestep; no dependence on past here
            # Reason: through self-attention trick, the last position/timestep already takes into account
            # all the past positions for every batch.
            logits = logits[:, -1, :]  # (B,vocab_size)
            # convert logits to probabilities
            probs = F.softmax(logits, dim=-1)
            # sample next token from probability distribution for every batch
            next_token = torch.multinomial(probs, num_samples=1)  # (B,1)
            # append next_token to the original context
            x = torch.cat((x, next_token), dim=1)  # (B,T+1)
        return x


# call the model
xb, yb = get_batch(split="train")

model = BigramModel()
m = model.to(device)
# number of parameters in the model
print(sum(p.numel() for p in m.parameters()) / 1e6, "M parameters")

## single forward pass
# logits, loss = m(x=xb, targets=yb)
# print(f"Loss = {loss}")

## model generation check
# model_output = model.generate(x=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=1000)
# print(decode(model_output[0].tolist()))

# -------------------------------train the model-------------------------------
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

for step in range(max_iters):
    xb, yb = get_batch("train")

    _, loss = model(x=xb, targets=yb)
    # set all gradients None
    optimizer.zero_grad(set_to_none=True)
    # run backward pass
    loss.backward()
    # make gradient update for each parameter
    optimizer.step()

    # print loss every few iterations
    if step % 1000 == 0:
        print(f"Iter: {step} | Loss: {loss.item()}")

print(f"Final Loss: {loss.item()}")

context = torch.zeros((1, 1), dtype=torch.long, device=device)
# model_output = m.generate(x=context, max_new_tokens=2000)
# print(decode(model_output[0].tolist()))
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


0.006793 M parameters
Iter: 0 | Loss: 4.097550868988037
Iter: 1000 | Loss: 2.8182942867279053
Iter: 2000 | Loss: 2.247951030731201
Iter: 3000 | Loss: 2.335516929626465
Iter: 4000 | Loss: 2.5938055515289307
Iter: 5000 | Loss: 2.5994932651519775
Iter: 6000 | Loss: 2.7757251262664795
Iter: 7000 | Loss: 2.3520689010620117
Iter: 8000 | Loss: 2.3567793369293213
Iter: 9000 | Loss: 2.227532386779785
Final Loss: 2.3101885318756104


I
HULUSL.

Who osgtris jundlle the-r weivive nol fath sneg if sifour:
Fomy orecavebre, my.

USHESA:
Bus whamoreent, slt prind yonge pth bied Igr sy fbar, ngrt otullll hmamy, bere alg vacirlt ofid Thyist hiupeme ofrees?

Bed that nfrighe, me'led stiend maxleath bekent ere; sounk er:
Corot,, tiln cous fe'pre wt hame ow
Mine ot ther yom oul the nn waven th,
Ctass? feat,
Y.
SE:
Go
IRjvecd fe to bre wet wae
Mck tn.
C3y mef rot at:
II kmft furalgns:
Tof a, rchee der.
NDUSEE:
O
S' Nlee I oth

Awo
I has 'net dlot tt tot woud o! blre fr
Wheillor rit wont weras,
I SHe sy ad, 

### Adding multi-attention head, blocks, skip connections, layernorm, dropout